<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks%20/%2004_diavigil_algorithmic_fairness_and_subgroup_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup & Core Deserialization**

In [ ]:
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Re-declare custom classes in __main__ for joblib unpickling
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)',
    'elix_peripheral_vascular': r'^(440|441|442|443|444|4471|5571|5579|V434)',
    'elix_hypertension_uncomplicated': r'^(401)',
    'elix_hypertension_complicated': r'^(402|403|404|405)',
    'elix_diabetes_complicated': r'^(250[4-9])',
    'elix_renal_failure': r'^(585|586|V420|V451|V56)',
    'elix_fluid_electrolyte': r'^(276)',
    'elix_chronic_pulmonary': r'^(490|491|492|493|494|495|496|500|501|502|503|504|505)',
    'elix_coagulopathy': r'^(286|2871|2873|2874|2875)'
}

def extract_elixhauser_flags(df: pd.DataFrame) -> pd.DataFrame:
    diag_cols = ['diag_1', 'diag_2', 'diag_3']
    comorb_features = pd.DataFrame(index=df.index)
    cleaned_diags = df[diag_cols].fillna('').astype(str).apply(
        lambda col: col.str.strip().str.replace('.', '', regex=False)
    )
    for flag_name, pattern in ELIXHAUSER_ICD9_RULES.items():
        compiled_regex = re.compile(pattern)
        matches = cleaned_diags.apply(
            lambda row: any(bool(compiled_regex.match(val)) for val in row if val),
            axis=1
        )
        comorb_features[flag_name] = matches.astype(int)
    return comorb_features

class ClinicalDomainFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medication_columns = [
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
            'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
            'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone'
        ]
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_out = X.copy()
        comorb_df = extract_elixhauser_flags(X_out)
        for col in comorb_df.columns:
            X_out[col] = comorb_df[col].values
        X_out['total_elixhauser_burden'] = comorb_df.sum(axis=1).values
        X_out['healthcare_utilization_index'] = (
            X_out['number_outpatient'].astype(float) +
            (2.0 * X_out['number_emergency'].astype(float)) +
            (3.0 * X_out['number_inpatient'].astype(float))
        )
        active_meds = X_out[self.medication_columns].apply(lambda row: (row != 'No').sum(), axis=1)
        med_titrations = X_out[self.medication_columns].apply(lambda row: row.isin(['Up', 'Down']).sum(), axis=1)
        X_out['active_diabetes_med_count'] = active_meds
        X_out['dosage_titration_count'] = med_titrations
        X_out['on_insulin'] = (X_out['insulin'] != 'No').astype(int)
        X_out['encounter_severity_index'] = (
            X_out['time_in_hospital'].astype(float) *
            (X_out['num_medications'].astype(float) / (X_out['num_lab_procedures'].astype(float) + 1.0))
        )
        X_out['glycemic_status_code'] = X_out['A1Cresult'].map({
            'None': 0, 'Norm': 1, '>7': 2, '>8': 2
        }).fillna(0).astype(int)
        return X_out

# Load the untouched audit test cohort
test_df = pd.read_parquet(os.path.join(DATA_PROCESSED, 'audit_test_cohort.parquet'))
artifacts = joblib.load(os.path.join(DATA_PROCESSED, 'model_calibration_artifacts.joblib'))

pipeline = artifacts['fitted_pipeline']
calibrator = artifacts['calibrator']
optimal_threshold = artifacts['optimal_threshold']

print(f"Loaded Untouched Audit Test Cohort: {len(test_df):,} encounters")
print(f"Loaded Optimal Threshold (tau*): {optimal_threshold:.4f}")